# Corpus-regression SL

In [ ]:
import polars as pl
import torch

from src import get_repo_base
from src.experiments.corpus_regression.config import data_dir
from src.experiments.corpus_regression.sl import CorpusRegressionSLConfig
from src.experiments.corpus_regression.utils import dim_averaged_metrics_from_parquet

repo_root = get_repo_base()
device = torch.device("cuda:0")

### Configs

In [ ]:
config = CorpusRegressionSLConfig.get_canonical(
    dataset_base_folder=data_dir(),
    study_base_folder=repo_root / "artifacts" / "corpus-regression-sl-example",
    num_lookforward_tokens=4,
    train_epochs=2,
)

display(config.visualize())

In [3]:
state = config.initialize(device=device)
state.run_training()

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

sl epoch 0:   0%|          | 0/781 [00:00<?, ?it/s]

validation epoch 0:   0%|          | 0/390 [00:00<?, ?it/s]

sl epoch 1:   0%|          | 0/781 [00:00<?, ?it/s]

validation epoch 1:   0%|          | 0/390 [00:00<?, ?it/s]

### Results

In [4]:
metrics = pl.read_parquet(config.study_folder / "metrics.parquet")
metrics

epoch,train_target_xx,train_target_xy,train_target_yy,train_target_n,val_target_xx,val_target_xy,val_target_yy,val_target_n
i64,list[f64],list[f64],list[f64],f64,list[f64],list[f64],list[f64],f64
0,"[3339.4729, 3560.268799, … 4402.100586]","[533.730896, 520.186951, … 56.092545]","[49984.0, 49984.0, … 49984.0]",49984.0,"[1992.052002, 507.727509, … 84.213951]","[-845.054871, 393.393372, … -47.864117]","[49920.0, 49920.0, … 49920.0]",49920.0
1,"[1835.387573, 1757.242554, … 1387.50415]","[731.521606, 904.023621, … 515.321228]","[49984.0, 49984.0, … 49984.0]",49984.0,"[3330.477539, 1552.287231, … 2178.428467]","[1141.810425, 777.154358, … 563.629578]","[49920.0, 49920.0, … 49920.0]",49920.0


In [5]:
dim_averaged_metrics_from_parquet(metrics, split="train").join(
    dim_averaged_metrics_from_parquet(metrics, split="val"), on="epoch"
)

epoch,train_avg_corr,train_avg_rsq,val_avg_corr,val_avg_rsq
i64,f64,f64,f64,f64
0,0.036316,-0.050994,0.036189,-0.011917
1,0.085772,-0.00312,0.060927,-0.019215


In [ ]:
last_epoch = int(metrics["epoch"].max())
validation_df = pl.read_parquet(
    config.study_folder / str(last_epoch) / "validation.parquet"
)
validation_df.head()

model_preds,target
list[f64],list[f64]
"[-0.196289, 0.100586, … -0.318359]","[-1.0, 1.0, … -1.0]"
"[-0.275391, 0.1796875, … -0.22168]","[-1.0, 1.0, … 1.0]"
"[-0.238281, 0.171875, … -0.185547]","[-1.0, 1.0, … -1.0]"
"[-0.138672, -0.047607, … -0.103516]","[1.0, -1.0, … 1.0]"
"[-0.152344, 0.135742, … -0.102051]","[-1.0, 1.0, … -1.0]"


: 